In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits, load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [162]:
TEST_SIZE = 0.3
CV_NUM_FOLDS = 10
RANDOM_STATE = 42

# 1. Choose datasets

In [163]:
# load datasets
digits = load_digits()
wine = load_wine()
cancer = load_breast_cancer()

# 2. Five models

In [239]:
algorithms = ['kNN', 'SVM', 'Naive Bayes', 'ID3', 'CART']
datasets = {'Digits': digits, 'Wine': wine, 'Breast Cancer': cancer}
results = {d: {a: 0 for a in algorithms} for d in datasets}

In [ ]:
def preprocess(dataset, normalize):
    '''For numpy array'''

    # separate features and target
    X, y = dataset.data, dataset.target

    # 70/30 train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    # feature normalization if needed
    if normalize:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

    return (X_train_scaled, X_test_scaled, y_train, y_test) if normalize else (X_train, X_test, y_train, y_test)


> ### i) kNN
> K-nearest neighbors

In [240]:
for name, dataset in datasets.items():
    X_train_scaled, X_test_scaled, y_train, y_test = preprocess(dataset, normalize=True)

    # try geometrically-spaced k-values
    k_values = np.geomspace(3, int(np.sqrt(len(X_train_scaled))), num=10)
    k_values = np.unique(k_values.astype(int))
    k_values = k_values[k_values % 2 == 1]  # keep odd only

    # initialization
    best_k = 3
    best_accuracy = 0

    # test different k values using cross-validation and find the best accuracy
    for k in k_values:
        knn_temp = KNeighborsClassifier(n_neighbors=k)
        kf = StratifiedKFold(n_splits=CV_NUM_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        scores = cross_val_score(knn_temp, X_train_scaled, y_train, cv=kf, scoring='accuracy')
        if scores.mean() > best_accuracy:
            best_k, best_accuracy = k, scores.mean()

    # fit and predict using the best k
    knn = KNeighborsClassifier(n_neighbors=best_k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    results[name]['kNN'] = accuracy
    print(best_k, 'is the best k for kNN on', name, 'dataset with test accuracy', round(accuracy, 2))

3 is the best k for kNN on Digits dataset with test accuracy 0.97
5 is the best k for kNN on Wine dataset with test accuracy 0.94
3 is the best k for kNN on Breast Cancer dataset with test accuracy 0.95


> ### ii) SVM
> Support vector machine

In [241]:
for name, dataset in datasets.items():
    X_train_scaled, X_test_scaled, y_train, y_test = preprocess(dataset, normalize=True)

    kernels = ['linear', 'rbf', 'poly']
    best_kernel = 'linear'
    best_accuracy = 0

    # test different kernels using cross-validation and find the best accuracy
    for kernel in kernels:
        svm_temp = SVC(kernel=kernel, random_state=RANDOM_STATE)
        kf = StratifiedKFold(n_splits=CV_NUM_FOLDS, shuffle=True, random_state=RANDOM_STATE)
        scores = cross_val_score(svm_temp, X_train_scaled, y_train, cv=kf, scoring='accuracy')
        if scores.mean() > best_accuracy:
            best_kernel, best_accuracy = kernel, scores.mean()

    # fit and predict using the best kernel
    svm = SVC(kernel=best_kernel, random_state=RANDOM_STATE)
    svm.fit(X_train_scaled, y_train)
    y_pred = svm.predict(X_test_scaled)
    accuracy = accuracy_score(y_test, y_pred)
    results[name]['SVM'] = accuracy
    print(best_kernel, 'is the best SVM kernel for', name, 'dataset with accuracy', round(accuracy, 2))

rbf is the best SVM kernel for Digits dataset with accuracy 0.98
rbf is the best SVM kernel for Wine dataset with accuracy 0.98
linear is the best SVM kernel for Breast Cancer dataset with accuracy 0.98


> ### iii) Naive Bayes

In [242]:
for name, dataset in datasets.items():
    X_train, X_test, y_train, y_test = preprocess(dataset, normalize=False)
    nb = GaussianNB()
    nb.fit(X_train, y_train)
    y_pred = nb.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name]['Naive Bayes'] = accuracy
    print('Naive Bayes for', name, 'dataset has accuracy', round(accuracy, 2))


Naive Bayes for Digits dataset has accuracy 0.82
Naive Bayes for Wine dataset has accuracy 1.0
Naive Bayes for Breast Cancer dataset has accuracy 0.95


> ### iv) ID3

In [243]:
for name, dataset in datasets.items():
    X_train, X_test, y_train, y_test = preprocess(dataset, normalize=False)
    id3 = DecisionTreeClassifier(criterion='entropy', random_state=RANDOM_STATE)
    id3.fit(X_train, y_train)
    y_pred = id3.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name]['ID3'] = accuracy
    print('ID3 for', name, 'dataset has accuracy', round(accuracy, 2))

ID3 for Digits dataset has accuracy 0.83
ID3 for Wine dataset has accuracy 0.91
ID3 for Breast Cancer dataset has accuracy 0.95


> ### v) CART

In [244]:
for name, dataset in datasets.items():
    X_train, X_test, y_train, y_test = preprocess(dataset, normalize=False)
    cart = DecisionTreeClassifier(criterion='gini', random_state=RANDOM_STATE)
    cart.fit(X_train, y_train)
    y_pred = cart.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    results[name]['CART'] = accuracy
    print('CART for', name, 'dataset has accuracy', round(accuracy, 2))

CART for Digits dataset has accuracy 0.85
CART for Wine dataset has accuracy 0.96
CART for Breast Cancer dataset has accuracy 0.92


In [245]:
print('ACCURACY')
pd.DataFrame(results)

ACCURACY


,Digits,Wine,Breast Cancer
kNN,0.972222,0.944444,0.953216
SVM,0.983333,0.981481,0.982456
Naive Bayes,0.822222,1.000000,0.947368
ID3,0.827778,0.907407,0.947368
CART,0.851852,0.962963,0.918129


# 3. Augmentation

In [ ]:
augmented_datasets = {name: {f'{name}_{i}': None for i in range(1, 6)} for name in datasets}
for name, dataset in datasets.items():
    df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
    df['target'] = dataset.target
    augmented_datasets[name][f'{name}_1'] = df.copy()  # original dataset

noise_scale = 0.05  # 5% noise
for name, dataset in datasets.items():
    df = pd.DataFrame(data=dataset.data, columns=dataset.feature_names)
    df['target'] = dataset.target
    len_ = len(df)
    for i in range(2, 6):
        n_samples = len_ * (i - 1)  # number of samples to add
        class_counts = df['target'].value_counts(normalize=True)
        temp_df = df.copy()
        for label in df['target'].unique():
            class_df = df[df['target'] == label]
            n_class_samples = int(n_samples * class_counts[label])  # get the proportion
            for _ in range(n_class_samples):
                row = class_df.sample(1, random_state=RANDOM_STATE).iloc[0].copy()
                for col in dataset.feature_names:
                    std = class_df[col].std()
                    noise = np.random.normal(0, noise_scale * std)
                    row[col] += noise
                temp_df = pd.concat([temp_df, pd.DataFrame([row])], ignore_index=True)
        augmented_datasets[name][f'{name}_{i}'] = temp_df

In [139]:
[augmented_datasets[x].keys() for x in augmented_datasets.keys()]

[dict_keys(['Digits_1', 'Digits_2', 'Digits_3', 'Digits_4', 'Digits_5']),
 dict_keys(['Wine_1', 'Wine_2', 'Wine_3', 'Wine_4', 'Wine_5']),
 dict_keys(['Breast Cancer_1', 'Breast Cancer_2', 'Breast Cancer_3', 'Breast Cancer_4', 'Breast Cancer_5'])]

# 4. Gradient boosting

In [223]:
boost_algos = ['XGBoost', 'CATBoost', 'LightGBM']
boost_results = {f'{name}_{i}': {} for i in range(1, 6) for name in datasets.keys()}

In [ ]:
def preprocess_df(dataset, normalize):
    '''For pandas dataframe'''

    # separate features and target
    feature_names = dataset.columns[:-1]
    target_name = dataset.columns[-1]
    X = dataset[feature_names].values
    y = dataset[target_name].values

    # 70/30 train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
    )

    # feature normalization if needed
    if normalize:
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

    return (X_train_scaled, X_test_scaled, y_train, y_test) if normalize else (X_train, X_test, y_train, y_test)


> ### i) XGBoost

In [249]:
for name, dataset_dict in augmented_datasets.items():
    for i in range(1, 6):
        dataset = dataset_dict[f'{name}_{i}']
        X_train, X_test, y_train, y_test = preprocess_df(dataset, normalize=False)
        xgb = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=RANDOM_STATE, verbosity=0)
        xgb.fit(X_train, y_train)
        y_pred = xgb.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        boost_results[f'{name}_{i}']['XGBoost'] = accuracy
        print('XGBoost for', name, 'dataset', i, 'has accuracy', round(accuracy, 2))
    print()

XGBoost for Digits dataset 1 has accuracy 0.95
XGBoost for Digits dataset 2 has accuracy 0.99
XGBoost for Digits dataset 3 has accuracy 0.98
XGBoost for Digits dataset 4 has accuracy 0.99
XGBoost for Digits dataset 5 has accuracy 0.99

XGBoost for Wine dataset 1 has accuracy 1.0
XGBoost for Wine dataset 2 has accuracy 0.94
XGBoost for Wine dataset 3 has accuracy 0.99
XGBoost for Wine dataset 4 has accuracy 0.99
XGBoost for Wine dataset 5 has accuracy 0.99

XGBoost for Breast Cancer dataset 1 has accuracy 0.96
XGBoost for Breast Cancer dataset 2 has accuracy 0.98
XGBoost for Breast Cancer dataset 3 has accuracy 0.98
XGBoost for Breast Cancer dataset 4 has accuracy 0.99
XGBoost for Breast Cancer dataset 5 has accuracy 0.99



> ### ii) CATBoost

In [248]:
for name, dataset_dict in augmented_datasets.items():
    for i in range(1, 6):
        dataset = dataset_dict[f'{name}_{i}']
        X_train, X_test, y_train, y_test = preprocess_df(dataset, normalize=False)
        cat = CatBoostClassifier(verbose=0, random_state=RANDOM_STATE)
        cat.fit(X_train, y_train)
        y_pred = cat.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        boost_results[f'{name}_{i}']['CATBoost'] = accuracy
        print('CATBoost for', name, 'dataset', i, 'has accuracy', round(accuracy, 2))
    print()

CATBoost for Digits dataset 1 has accuracy 0.97
CATBoost for Digits dataset 2 has accuracy 0.99
CATBoost for Digits dataset 3 has accuracy 0.99
CATBoost for Digits dataset 4 has accuracy 1.0
CATBoost for Digits dataset 5 has accuracy 1.0

CATBoost for Wine dataset 1 has accuracy 0.98
CATBoost for Wine dataset 2 has accuracy 0.98
CATBoost for Wine dataset 3 has accuracy 0.99
CATBoost for Wine dataset 4 has accuracy 1.0
CATBoost for Wine dataset 5 has accuracy 1.0

CATBoost for Breast Cancer dataset 1 has accuracy 0.95
CATBoost for Breast Cancer dataset 2 has accuracy 0.98
CATBoost for Breast Cancer dataset 3 has accuracy 0.98
CATBoost for Breast Cancer dataset 4 has accuracy 1.0
CATBoost for Breast Cancer dataset 5 has accuracy 0.99



> ### iii) LightGBM

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

for name, dataset_dict in augmented_datasets.items():
    for i in range(1, 6):
        dataset = dataset_dict[f'{name}_{i}']
        X_train, X_test, y_train, y_test = preprocess_df(dataset, normalize=False)
        lgb = LGBMClassifier(random_state=RANDOM_STATE, verbose=-1)
        lgb.fit(X_train, y_train)
        y_pred = lgb.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        boost_results[f'{name}_{i}']['LightGBM'] = accuracy
        print('LightGBM for', name, 'dataset', i, 'has accuracy', round(accuracy, 2))
    print()

LightGBM for Digits dataset 1 has accuracy 0.96
LightGBM for Digits dataset 2 has accuracy 0.99
LightGBM for Digits dataset 3 has accuracy 0.99
LightGBM for Digits dataset 4 has accuracy 0.99
LightGBM for Digits dataset 5 has accuracy 0.99

LightGBM for Wine dataset 1 has accuracy 0.98
LightGBM for Wine dataset 2 has accuracy 0.98
LightGBM for Wine dataset 3 has accuracy 0.99
LightGBM for Wine dataset 4 has accuracy 1.0
LightGBM for Wine dataset 5 has accuracy 0.99

LightGBM for Breast Cancer dataset 1 has accuracy 0.96
LightGBM for Breast Cancer dataset 2 has accuracy 0.97
LightGBM for Breast Cancer dataset 3 has accuracy 0.98
LightGBM for Breast Cancer dataset 4 has accuracy 0.99
LightGBM for Breast Cancer dataset 5 has accuracy 0.99



In [251]:
print('ACCURACY')
pd.DataFrame(boost_results).T.sort_index()

ACCURACY


,XGBoost,LightGBM,CATBoost
Breast Cancer_1,0.964912,0.959064,0.953216
Breast Cancer_2,0.982456,0.973684,0.976608
Breast Cancer_3,0.982456,0.978558,0.982456
Breast Cancer_4,0.992679,0.989751,0.995608
Breast Cancer_5,0.992974,0.994145,0.991803
Digits_1,0.948148,0.957407,0.974074
Digits_2,0.985171,0.987025,0.989805
Digits_3,0.982695,0.987639,0.990729
Digits_4,0.989801,0.993510,0.995364
Digits_5,0.991098,0.992211,0.995178


In [252]:
print('Execution times')
print('XGBoost: 4.7 seconds')
print('CATBoost: 66.3 seconds')
print('LightGBM: 12.1 seconds')

Execution times
XGBoost: 4.7 seconds
CATBoost: 66.3 seconds
LightGBM: 12.1 seconds
